# Task 12 — Scope, Iterators, Generators & Decorators

This task sheet is the hands-on companion to Session 12. It carries ten questions grouped into three themes, and each group directly reuses the theory you just studied. The first theme is **namespace and scope**. Q1 and Q2 ask you to build a `Person` class and then expose its namespace from two sides — the class itself and an instance of it. Q3 switches to recursion, asking for the greatest common divisor plus a count of how many function calls the recursion performed. The second theme is **iterators and generators**. Q4 asks you to rebuild Python's built-in `enumerate` as a class called `MyEnumerate`. Q5 stretches it: a `Circle` that cycles through a sequence a fixed number of times, repeating when the count exceeds the sequence length — perfect practice for the `__iter__` / `__next__` protocol. Q6 asks for `elapsed_since`, a generator that times each iteration relative to the previous one, which forces you to think about how generators pause and resume. The third theme is **decorators**. Q7 chains three HTML-tag decorators (`bold`, `italic`, `underline`) around one function — order matters and will teach you bottom-up composition. Q8 builds a `printer` decorator that echoes returned values, Q9 a `twice` decorator that calls a function twice, and Q10 a `double` decorator plus an `assert` that proves it works.

Read each question carefully, then the short concept-and-hints cell underneath it. Try to write the solution yourself before reading the hints. Keep your code readable, and when a question mentions a protocol — iterator, generator, decorator — recall the exact dunder methods or the `yield` keyword that the theory cells describe.

# 1. Namespace and Scope

## 1.1 Where the names of a class live

From Session 12 you know that every name lives in a namespace — a dictionary mapping identifiers to objects. Classes and their instances each have their own. Verify this with the `__dict__` attribute:

- `Person.__dict__` is the **class namespace**: it holds the methods (`address`, and `__init__` if we define one) and any class-level attributes.
- `person.__dict__` is an **instance namespace**: it holds the instance attributes we set inside `__init__`, one entry per attribute.

The attribute model of Python is deliberately transparent: attributes, `__init__`, even the dunder methods are just entries in these dictionaries. Calling a method like `person.address()` is really `Person.address(person)` — Python searches the instance namespace, then the class, then its bases. That search order is the object-oriented cousin of the LEGB rule.

One more convention to remember: in this course "**private**" attributes do not get special enforcement — there are no true private fields in Python. We signal privacy by naming, with a single underscore (`_city`) for "internal use" and double underscores (`__age`) for name-mangled members. The class spec for Q1 marks `city` and `age` as private, so your `__init__` should store them with leading underscores.

### Hints for Q1
- Write `__init__(self, name, state, city, age)` and assign all four.
- The `address` method combines `name`, `city` and `state` into `"<name>, <city>, <state>"` — look at the parametername order in the spec.
- Display the namespace with `print(Person.__dict__)`.

###`Q1:` Write `Person` Class as given below and then display it's namespace.



```
Class Name - Person

Attributes:
name - public
state - public
city - private
age - private

Methods:
address - public
It give address of the person as "<name>, <city>, <state>"
```

### Q1 — checking your work

After creating the class and an instance, your expected behaviour is:

```python
p = Person('Amar', 'Delhi', 'New Delhi', 24)
print(p.address())   # Amar, New Delhi, Delhi
```

Plan the three checks:
1. `p.address()` returns the composed string — verify the attribute order exactly matches the spec.
2. `Person.__dict__` shows `address` and `__init__` (plus dunders) — the class namespace.
3. `p.__dict__` shows exactly your four stored attributes.

If `address` returns the wrong format, the usual culprit is mixing the `city` / `state` order or missing the comma-and-space separator.

In [ ]:
#Write your code here

###`Q2:` Write a program to show namespace of object/instance of above(Person) class.

### Q2 — the instance namespace

This question is deliberately short: after building a `Person` instance in the previous cell, show the namespace of that *object*. In Python, an instance's attributes live in `obj.__dict__`:

```python
p = Person('Amar', 'Delhi', 'New Delhi', 24)
print(p.__dict__)
```

Your output should be a dictionary with exactly the keys you stored in `__init__` — the public `name` and `state`, and the underscore-prefixed private `_city` and `_age`. This difference between the two namespace cells is the takeaway: the class namespace holds shared *behaviour* (methods), while each instance namespace holds *state* (that object's own values). Two instances of the same class have the same class namespace but completely different instance dictionaries — the reason that every object carries its own data.

In [ ]:
# Write your code here


###`Q3:` Write a recursive program to to calculate `gcd` and print no. of function calls taken to find the solution.
```
gcd(5,10) -> result in 5 as gcd and function call 4
```

### Q3 — counting calls inside recursion

Finding the GCD by recursion is the Euclid method: `gcd(a, b)` returns `a` when `b == 0`, otherwise `gcd(b, a % b)`. The twist is the call counter — every time the function *enters* (including the first call from the module level), we add one.

The cleanest way to count without a global variable is a small closure:

```python
def gcd(a, b):
    count = 0
    def step(x, y):
        nonlocal count
        count += 1
        return (x if y == 0 else step(y, x % y)), count
    return step(a, b)
```

or simpler, a module-level counter or a function attribute such as `gcd.calls += 1`. For `gcd(5, 10)`: the recursion shrinks `(5,10) → (10,5) → (5,0)`, which is **three** steps of the algorithm plus... count carefully; the expected output in the question says the function is called **4** times in total. Trace exactly how many *entries* your function performs and match it to the spec before moving on.

In [ ]:
#Write your code here

## Itterator And Generator

# 2. Iterators and Generators

## 2.1 The iteration protocol

A **for loop** does not understand lists, strings, or dicts specially — it works with anything that follows the *iterator protocol*:

1. `iter(obj)` returns an iterator object (or raises `TypeError` if the object is not iterable).
2. The iterator's `__next__()` returns the next element each time it is called.
3. When elements run out, `__next__()` raises `StopIteration`, and the loop gracefully ends.

You can build your own iterable by writing a class with `__iter__` returning an iterator, and an iterator class with `__next__`. This is exactly how `for index, letter in MyEnumerate('abc')` will work in Q4.

**Generators** automate the same protocol: any function containing the `yield` keyword is a generator. Calling it returns a *generator object*; each `next()` resumes execution from the previous `yield`, freezing all local variables in between. No `__next__` class is needed — the machinery is built in. `yield` is the lazy cousin of `return`: the body does not run until someone iterates, and it can produce values indefinitely without storing them all.

### Hints for Q4–Q5
- `__iter__` should return the iterator object (often `self`); `__next__` does the stepping.
- To reject a non-iterable in Q4, call `iter(data)` in `__init__` — it raises `TypeError` for you; store it and call `next()` on it inside `__next__`.
- For the cycling `Circle` in Q5, keep an index and use `index % len(sequence)`; stop after the requested number of elements.

###`Q4:` Create MyEnumerate class, 
Create your own `MyEnumerate` class such that someone can use it instead of enumerate. It will need to return a `tuple` with each iteration, with the first element in the tuple being the `index` (starting with 0) and the second element being the `current element` from the underlying data structure. Trying to use `MyEnumerate` with a noniterable argument will result in an error.

```
for index, letter in MyEnumerate('abc'):
    print(f'{index} : {letter}')
```

Output:
```
0 : a
1 : b
2 : c
```

### Q4 — MyEnumerate

Reproduce `enumerate` with three pieces of state: a `stop` threshold, an index that starts at `0`, and an iterator over the data. A compact implementation:

```python
class MyEnumerate:
    def __init__(self, data):
        self._data = iter(data)   # TypeError for non-iterables, naturally
        self._index = 0

    def __iter__(self):
        return self

    def __next__(self):
        value = next(self._data)
        result = (self._index, value)
        self._index += 1
        return result
```

Study the shape: `__iter__` returns `self` so the object is both iterable and its own iterator; `__next__` performs one step and returns a **tuple** (index, element); running out means `next(self._data)` raises `StopIteration`, which propagates and ends the for loop. Test that `list(MyEnumerate('abc'))` equals `[(0, 'a'), (1, 'b'), (2, 'c')]`, and confirm that passing a non-iterable like `5` raises `TypeError`.

In [ ]:
#Write your code here

0 : a
1 : b
2 : c


###`Q5:` Iterate in circle
Define a class, `Circle`, that takes two arguments when defined: a sequence and a number. The idea is that the object will then return elements the defined number of times. If the number is greater than the number of elements, then the sequence  repeats as necessary. You can define an another class used as a helper (like I call `CircleIterator`).

```
c = Circle('abc', 5)
d = Circle('abc', 7)
print(list(c))
print(list(d))
```

Output
```
[a, b, c, a, b]
[a, b, c, a, b, c, a]
```

### Q5 — Circle: cycling with a bound

`Circle` needs a helper iterator (the question suggests `CircleIterator`) plus the two arguments, and it must stop after *exactly* `number` elements, repeating the sequence as needed. The trick that makes cycling trivial is modulo arithmetic:

```python
class CircleIterator:
    def __init__(self, seq, number):
        self.seq = seq
        self.number = number
        self.i = 0

    def __next__(self):
        if self.i >= self.number:
            raise StopIteration
        value = self.seq[self.i % len(self.seq)]
        self.i += 1
        return value

    def __iter__(self):
        return self

class Circle:
    def __init__(self, seq, number):
        self.seq, self.number = seq, number
    def __iter__(self):
        return CircleIterator(self.seq, self.number)
```

For `Circle('abc', 5)` the sequence `'abc'` is only three long, so the wrap-around gives `a, b, c, a, b`. `print(list(c))` then produces the exact expected output. The `len` guard also protects the edge case where `number` is missing or smaller than `len` — modulo still works fine.

In [ ]:
#Write your code here


['a', 'b', 'c', 'a', 'b']
['a', 'b', 'c', 'a', 'b', 'c', 'a']


['a', 'b', 'c', 'a', 'b']


###`Q6:` Generator time elapsed
Write a generator function whose argument must be iterable. With each iteration, the generator will return a two-element tuple. The first element in the tuple will be an integer indicating how many seconds have passed since the previous iteration. The tuple’s second element will be the next item from the passed argument.

Note that the timing should be relative to the previous iteration, not when the
generator was first created or invoked. Thus the timing number in the first iteration
will be 0

```
for t in elapsed_since('abcd'):
    print(t)
    time.sleep(2)
```

Output:
```
(0.0, 'a')
(2.005651817999933, 'b')
(2.0023095009998997, 'c')
(2.001949742000079, 'd')
```
Note: Your output may differ because of diffrent system has different processing configuration.

### Q6 — elapsed_since generator

This generator measures *clock* time between consecutive iterations. Because you need `yield`, the function body is naturally lazy: nothing runs until the loop asks for the first value.

```python
import time

def elapsed_since(data):
    last = None
    for item in data:
        now = time.time()
        if last is None:
            yield (0.0, item)
        else:
            yield (now - last, item)
        last = now
```

The empty-first trick: on the first iteration the previous timestamp does not exist, so `elapsed == 0`, matching the spec's `(0.0, 'a')`. Every later `yield` computes the difference between the current clock and the previous one, so the timing is **relative to the previous iteration**, not to the moment the generator was created. Run the sample loop with `time.sleep(2)` and confirm each gap is about 2 seconds; small differences between systems are expected.

In [ ]:
#Write yor code

(0.0, 'a')
(2.005651817999933, 'b')
(2.0023095009998997, 'c')
(2.001949742000079, 'd')


## Decorators

# 3. Decorators

## 3.1 Wrapping behaviour around a function

A decorator, from Session 12, is a function that takes a function and returns a wrapped version: `@deco` is sugar for `func = deco(func)`. The core pattern is always the same — an outer function that closes over the original and returns a closure:

```python
def deco(func):
    def wrapper(*args, **kwargs):
        # before
        result = func(*args, **kwargs)
        # after
        return result
    return wrapper
```

The wrapper sees the original through the closure and can run code before it, after it, or replace its result altogether — which is exactly what Q7–Q10 exploit.

One detail decides several of these questions: decorators are applied **bottom-up**. For

```python
@bold
@italic
@underline
def hello():
    return "hello world"
```

Python builds `hello = bold(italic(underline(hello)))`. When called, `bold`'s wrapper runs first, then `italic`'s, then `underline`'s, then the original — innermost decoration executes last, closest to the function. Reading the output `<b><i><u>hello world</u></i></b>` from the inside out confirms exactly that nesting order.

###`Q7:` Write a Python program to make a chain of function decorators (bold, italic, underline etc.) on a given function which prints "hello world"

```
def hello():
    return "hello world"
```

```
bold - wrap string with <b> tag. <b>Str</b>
italic - wrap string with <i> tag. <i>Str</i>
underline- wrap string with <u> tag. <u>Str</u>
```

### Q7 — a chain of optional decorators: bold, italic, underline

Write three near-identical decorators, one per HTML tag. Each returns a wrapper that wraps the returned string in its tag:

```python
def bold(func):
    def wrapper():
        return '<b>' + func() + '</b>'
    return wrapper
def italic(func):
    def wrapper():
        return '<i>' + func() + '</i>'
    return wrapper
def underline(func):
    def wrapper():
        return '<u>' + func() + '</u>'
    return wrapper
```

Stack all three above the plain `hello` (which simply returns `"hello world"`). Remember the bottom-up application, so the expected result is `<b><i><u>hello world</u></i></b>` — bold outermost, underline innermost. Printing the decorated call and comparing to that exact string is your correctness check. Notice this works because `hello` *returns* its text; the wrappers must return too, never print, so the tags nest.

In [ ]:
#Write your code here

<b><i><u>hello world</u></i></b>


###`Q8:` Write a decorator called `printer` which causes any decorated function to print their return values. If the return value of a given function is `None`, printer should do nothing.



### Q8 — the printer decorator

The `printer` decorator prints whatever the wrapped function returns — unless it returns `None`, in which case it stays silent. The wrapper must **capture the return value before printing**:

```python
def printer(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        if result is not None:
            print(result)
        return result
    return wrapper
```

The `if result is not None` guard is the whole point of the exercise: a function like `def hello(): return 'abc'` will print `abc`, while `def nothing(): return None` prints nothing. And the wrapper still returns the original result, so decoration does not change the function's contract. Test with functions returning strings (`'abc'`, `'pqr'`) as the sample output shows.

In [ ]:
#Write your code here

abc
pqr


###`Q9:` Make a decorator which calls a given function twice. You can assume the functions don't return anything important, but they may take arguments.
```
#Lets say given function 
def hello(string):
    print(string)

#on calling after specified decorator is inplaced
hello('hello')
```

Output:
```
hello
hello
```

### Q9 — the twice decorator

This decorator calls the wrapped function twice. The functions "don't return anything important", so the wrapper can ignore return values entirely — it only needs to forward arguments:

```python
def twice(func):
    def wrapper(*args, **kwargs):
        func(*args, **kwargs)
        return func(*args, **kwargs)
    return wrapper

def hello(string):
    print(string)

hello('hello')   # prints hello twice
```

Two details to respect: the wrapper must accept arbitrary parameters (`*args, **kwargs`) because decorated functions may take arguments — here `string` — and it simply calls the original twice. The expected output is `hello` printed twice, once per call. If you forget the arguments in the signature, the decorator will fail whenever the function takes any input.

In [ ]:
#Write your cod here

hello
hello


### `Q10:` Write a decorator which doubles the return value of any function. And test that decoratos is working correctly or not using `asert`.

```
add(2,3) -> result in 10. Without decorator it should be 5.
```

### Q10 — the double decorator with an assert

This decorator multiplies the function *return value* by two, then the rest of the program proves it works with `assert`:

```python
def double(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs) * 2
    return wrapper

@double
def add(a, b):
    return a + b

assert add(2, 3) == 10   # 5 doubled
```

The arithmetic matters: `add(2, 3)` returns `5`, the wrapper multiplies it by `2` to get `10`, and `assert` silently passes when correct. `assert` raises `AssertionError` on failure, so it is the idiomatic quick check in problems like this. Without the decorator the same call would give `5` — one line of `@double` changes the contract, which is the decorative power you have been practising throughout this task.

## Summary

This task put Session 12's machinery to work in ten concrete programs. On namespaces, you exposed a class and an instance through `__dict__`, separating shared behaviour from per-object state, and wrote a recursive GCD that also counted its own calls. On iteration, you rebuilt `enumerate` as the `MyEnumerate` class using `__iter__`/`__next__`, designed a `Circle` whose helper iterator cycles a sequence with modular arithmetic, and wrote the `elapsed_since` generator that times each step relative to the previous one. On decorators, you chained HTML-tag wrappers in the right order, printed return values with `printer`, called functions twice with `twice`, and doubled results while verifying correctness with `assert`. Together the ten solutions cover the full journey of Session 12: knowing where names live, building iteration yourself, and wrapping functions with behaviour.

In [ ]:
# Write your code here
